# 02 — Preprocessing & Feature Engineering
**Customer Churn Prediction — Banking**

Goal: Prepare clean, model-ready features. Handle encoding, scaling, and class imbalance via SMOTE.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/Churn_Modelling.csv')
df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)
print(f'Loaded dataset: {df.shape}')
df.head()

## 1. Encode Categorical Variables

In [ ]:
# One-hot encode Geography (avoids ordinal assumption)
df = pd.get_dummies(df, columns=['Geography'], drop_first=False)

# Binary encode Gender
df['Gender'] = LabelEncoder().fit_transform(df['Gender'])  # Male=1, Female=0

print('Columns after encoding:')
print(df.columns.tolist())
df.head(3)

## 2. Feature Engineering

In [ ]:
# Balance to salary ratio — proxy for financial health
df['Balance_Salary_Ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)

# Products per tenure — engagement rate
df['Products_per_Tenure'] = df['NumOfProducts'] / (df['Tenure'] + 1)

# Age group buckets
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 30, 45, 60, 100],
                          labels=[0, 1, 2, 3]).astype(int)

print('New features added: Balance_Salary_Ratio, Products_per_Tenure, Age_Group')
print(f'Dataset shape after feature engineering: {df.shape}')
df[['Balance_Salary_Ratio', 'Products_per_Tenure', 'Age_Group']].describe().round(3)

## 3. Train/Test Split

In [ ]:
X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')
print(f'Train churn rate: {y_train.mean():.3f}')
print(f'Test churn rate:  {y_test.mean():.3f}')

## 4. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print('Features scaled using StandardScaler (fit on train only to prevent data leakage).')
X_train_scaled.describe().round(2)

## 5. Handle Class Imbalance with SMOTE

In [ ]:
print(f'Before SMOTE — Class distribution:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'\nAfter SMOTE — Class distribution:')
print(pd.Series(y_train_resampled).value_counts())
print(f'\nNew training size: {X_train_resampled.shape[0]}')

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['#457B9D', '#E63946'])
axes[0].set_title('Before SMOTE', fontweight='bold')
axes[0].set_xticklabels(['Retained', 'Churned'], rotation=0)

pd.Series(y_train_resampled).value_counts().plot(kind='bar', ax=axes[1], color=['#457B9D', '#E63946'])
axes[1].set_title('After SMOTE (Balanced)', fontweight='bold')
axes[1].set_xticklabels(['Retained', 'Churned'], rotation=0)

plt.tight_layout()
plt.savefig('../outputs/smote_balance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Processed Data

In [ ]:
import pickle

# Save all artifacts for use in modeling notebook
data_artifacts = {
    'X_train': X_train_resampled,
    'X_test': X_test_scaled,
    'y_train': y_train_resampled,
    'y_test': y_test,
    'feature_names': X.columns.tolist(),
    'scaler': scaler
}

with open('../data/processed_data.pkl', 'wb') as f:
    pickle.dump(data_artifacts, f)

print('Saved processed data to ../data/processed_data.pkl')
print(f'Features: {X.columns.tolist()}')

## Preprocessing Summary

| Step | Decision | Rationale |
|---|---|---|
| Encoding | One-hot for Geography, binary for Gender | Avoids ordinal assumption |
| Feature engineering | 3 new features added | Balance/salary ratio, engagement rate, age bucket |
| Scaling | StandardScaler on train set only | Prevents data leakage into test |
| Class imbalance | SMOTE on training set only | Accuracy is misleading at 80/20 split |

→ **Next step:** Model training and comparison in `03_models.ipynb`